# NEXUS AI — Paired Reader v5

Dean A. Kulik · QuHarmonics Research Group · ORCID 0009-0003-3128-8828

Native paired reader with exact inverse, lazy relative field and supplied-depth self-transport certificates.
This notebook adds independent paired tools without changing the single-reader v4.1 ledger. Run the next code cell once; it completes without waiting for input.


In [1]:
"""NEXUS local AI / compiler bridge — compiler-side BIND resolver / lazy frame v4.
Dean A. Kulik, QuHarmonics Research Group, ORCID 0009-0003-3128-8828.

Python 3.10+, standard library only. Runs as a script or in one Jupyter cell.
LM Studio: http://localhost:1235/v1 (override with LM_STUDIO_URL).

Conceptual state: (w,E,p) with E[r]=D[p XOR r].
Lazy field: E[r]=V[r XOR s], backing dictionary V and internal frame s.
F: x=V[s]^w; V'[s]=V[s]^P(x); s'=s^x; p'=p^x; w'=P(x).
H_h: p'=p^h, w'=w, (V',s')=(V,s).
Both forward and transport perform O(1) native field operations.
F H_h = H_h F, H_h^(-1)=H_h, H_a H_b=H_(a^b).
Old absolute snapshots and v1/v2/v3 hash-linked ledger records remain readable.
BIND derives one-step routes from exact readout constraints without traversing ring addresses.
BIND is read-only and never silently fills a missing target field.
Full ledger before/after snapshots still require O(active cells) serialization.

The local language model requests tools; the compiler computes and verifies operations.
No unrestricted shell or filesystem tool is exposed to the model.
"""
from __future__ import annotations
import hashlib
import json
import os
import random
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

BASE = os.getenv('LM_STUDIO_URL', 'http://localhost:1235/v1').rstrip('/')
if BASE.endswith('/v1'):
    BASE = BASE[:-3]
TOKEN = os.getenv('LM_API_TOKEN', '')
WIDTH, SEED_P, SEED_W = 4, 0, 1
ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
LEDGER = ROOT / 'NEXUS_local_AI_events.jsonl'  # Reuse & verify the original history.
GENESIS = '0' * 64
BIND_PREDICTION = 'NEXUS BIND v4 compiler-side prediction, 2026-11-02.\nBefore linking the next independent test readout:\n1. For any native width 2..16, any complete valid destination has one F-predecessor given by the exact inverse of F; a forward replay reconstructs the destination bit-for-bit.\n2. For any live state and target WHERE, the unique XOR transport h = p XOR target_where yields the same (w,E) and returns by applying h again.\n3. Given a complete one-step endpoint, the three declared provider contracts H, F, F^-1 are evaluated algebraically; zero providers -> UNBOUND, one -> BOUND, multiple -> AMBIGUOUS. Missing field -> UNLINKED and never assumed empty.\n4. BIND neither changes the live machine nor appends to the event ledger, and old ledger v1, v2, v3 events remain replayable.\nWAY HOME: BOUND-H returns by H_h; BOUND F-predecessor returns by F; BOUND-F returns by F^-1; BOUND-F^-1 returns by F; each is checked on the complete (p,w,D) state.\n'
BIND_PREDICTION_SHA256 = '65a5dae5b1d5543799c1700d8970149a1021c9c7618ab66ee27d58ea9e8bbda0'


def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False)


def chain_hash(previous, payload):
    return hashlib.sha256((previous + '\n' + canonical(payload)).encode('utf-8')).hexdigest()


def partner(x: int, n: int) -> int:
    """Native P: reverse nonpalindromes, complement palindromes."""
    mask = (1 << n) - 1
    if not (2 <= n <= 16 and 0 <= x <= mask):
        raise ValueError('n must be 2..16 and x must fit n bits')
    rev = int(f'{x:0{n}b}'[::-1], 2)
    return rev if rev != x else x ^ mask


class Machine:
    """Native conceptual state (w, E, p); lazy E[r]=V[r XOR s].

    V is the sparse backing field, s the internal frame, p the absolute reader WHERE.
    Both forward/backward and transport touch O(1) native field cells.
    Exporting a complete absolute/relative field costs O(k) for k occupied cells.
    """
    def __init__(self, n=WIDTH, p=SEED_P, w=SEED_W, field=None,
                 *, relative_field=None):
        if not (2 <= n <= 16):
            raise ValueError('n must be 2..16 for this prototype')
        mask = (1 << n) - 1
        if type(p) is not int or type(w) is not int or not (0 <= p <= mask and 0 <= w <= mask):
            raise ValueError('p and w must be n-bit words')
        if field is not None and relative_field is not None:
            raise ValueError('supply absolute field or relative_field, not both')
        # Initial frame is zero: the backing keys begin in reader-relative coordinates.
        if relative_field is not None:
            backing = {int(k): int(v) for k, v in relative_field.items() if int(v)}
        else:
            backing = {int(k) ^ p: int(v) for k, v in (field or {}).items() if int(v)}
        if any(not (0 <= k <= mask and 0 <= v <= mask) for k,v in backing.items()):
            raise ValueError('field keys and values must be n-bit words')
        self.n, self.p, self.w = n, p, w
        self.V, self.s = backing, 0
        self.last_hash = GENESIS

    def copy(self):
        other = Machine(self.n, self.p, self.w, relative_field={})
        other.V, other.s = self.V.copy(), self.s
        other.last_hash = self.last_hash
        return other

    def adopt(self, other):
        self.p, self.w = other.p, other.w
        self.V, self.s = other.V.copy(), other.s
        self.last_hash = other.last_hash

    def snapshot(self):
        """External v2-compatible state (p,w,D), for tool and ledger compatibility."""
        absolute = {self.p ^ self.s ^ k: v for k,v in self.V.items()}
        return {'n': self.n, 'p': self.p, 'w': self.w,
                'field': {str(k): absolute[k] for k in sorted(absolute)}}

    def relative_snapshot(self):
        relative = {k ^ self.s: v for k,v in self.V.items()}
        return {str(k): relative[k] for k in sorted(relative)}

    @staticmethod
    def _set(field, at, value):
        if value:
            field[at] = value
        else:
            field.pop(at, None)

    def forward(self):
        """One-cell native step, O(1) independent of field occupancy."""
        oldp, oldframe = self.p, self.s
        x = self.V.get(oldframe, 0) ^ self.w
        w2 = partner(x, self.n)
        self._set(self.V, oldframe, self.V.get(oldframe, 0) ^ w2)
        self.s, self.p, self.w = oldframe ^ x, oldp ^ x, w2
        return {'direction': 'forward', 'x': x, 'written_at': oldp}

    def backward(self):
        """One-cell exact inverse, independent of the number of occupied cells."""
        x = partner(self.w, self.n)
        oldframe, oldp = self.s ^ x, self.p ^ x
        self._set(self.V, oldframe, self.V.get(oldframe, 0) ^ self.w)
        oldw = x ^ self.V.get(oldframe, 0)
        self.s, self.p, self.w = oldframe, oldp, oldw
        return {'direction': 'reverse', 'x': x, 'restored_at': oldp}

    def transport(self, h: int):
        """O(1) pure placement; both V and s remain literally unchanged."""
        if type(h) is not int or not 0 <= h < (1 << self.n):
            raise ValueError(f'transport h must be an integer 0..{(1 << self.n)-1}')
        prev = self.p
        self.p ^= h
        # Keep exact v2 route metadata to preserve historical event-replay checks.
        return {'direction': 'transport', 'h': h, 'from_word': prev,
                'to_word': self.p, 'relocated_field_cells': len(self.V)}

def verify():
    """Executed deterministic checks: involution, random F/H commuting squares, routes."""
    checks = 0
    for n in range(2, 11):
        for x in range(1 << n):
            assert partner(partner(x, n), n) == x
            assert partner(x, n) != x
            checks += 1
    rng = random.Random(181)
    for n in range(2, 17):
        mask = (1 << n) - 1
        for _ in range(320):
            D = {}
            for __ in range(1 + rng.randrange(12)):
                k = rng.randrange(mask + 1)
                v = rng.randrange(mask + 1)
                if v:
                    D[k] = v
            m = Machine(n, rng.randrange(mask+1), rng.randrange(mask+1), D)
            h, a, b = (rng.randrange(mask+1) for __ in range(3))
            # H is its own inverse; transport group is XOR.
            t = m.copy(); original_V = t.V; original_s = t.s
            t.transport(h)
            assert t.V is original_V and t.s == original_s
            t.transport(h)
            assert t.V is original_V and t.s == original_s
            assert t.snapshot() == m.snapshot()
            t = m.copy(); t.transport(a); t.transport(b)
            u = m.copy(); u.transport(a ^ b)
            assert t.snapshot() == u.snapshot()
            # The commuting square F o H == H o F.
            left = m.copy(); left.transport(h); left.forward()
            right = m.copy(); right.forward(); right.transport(h)
            assert left.snapshot() == right.snapshot(), ('FH=HF', n)
            # Also commute for inverse steps.
            left = m.copy(); left.transport(h); left.backward()
            right = m.copy(); right.backward(); right.transport(h)
            assert left.snapshot() == right.snapshot(), ('F^-1H=HF^-1', n)
            # Roundtrip on arbitrary populated field, not just vacuum seed.
            t = m.copy(); t.forward(); t.backward()
            assert t.snapshot() == m.snapshot()
            t = m.copy(); t.backward(); t.forward()
            assert t.snapshot() == m.snapshot()
            checks += 6
    for n in (4, 8, 16):
        m = Machine(n=n, p=0, w=1)
        initial = m.snapshot()
        for _ in range(1000):
            m.forward()
        for _ in range(1000):
            m.backward()
        assert m.snapshot() == initial
    print(f'RUNS: {checks:,} deterministic conditions; P, arbitrary-field roundtrip, '
          'transport involution/group law, F H = H F, inverse commutation, '
          'and 1000-step return at n=4,8,16.')


def apply_mutation(machine, kind, h=None):
    if kind == 'step':
        return machine.forward()
    if kind == 'reverse':
        return machine.backward()
    if kind == 'transport':
        return machine.transport(h)
    raise ValueError('unknown operation')


def replay():
    m = Machine()
    if not LEDGER.exists():
        return m
    count, legacy = 0, 0
    with LEDGER.open('r', encoding='utf-8') as f:
        for count, line in enumerate(f, 1):
            event = json.loads(line)
            if event['before'] != m.snapshot():
                raise RuntimeError(f'Ledger discontinuity before line {count}')
            kind = event['operation']
            # Supports old unhashed events without rewriting or discarding them.
            if 'event_hash' in event:
                if event.get('prev_hash') != m.last_hash:
                    raise RuntimeError(f'Ledger hash link invalid at line {count}')
                payload = {k: v for k, v in event.items()
                           if k not in ('prev_hash', 'event_hash')}
                digest = chain_hash(m.last_hash, payload)
                if digest != event['event_hash']:
                    raise RuntimeError(f'Ledger event hash invalid at line {count}')
            else:
                legacy += 1
                payload = event
                digest = chain_hash(m.last_hash, payload)
            meta = apply_mutation(m, kind, event.get('route', {}).get('h'))
            if event['after'] != m.snapshot() or (event.get('route') and meta != event['route']):
                raise RuntimeError(f'Ledger event mismatch after line {count}')
            m.last_hash = digest
    print(f'Replayed {count} event(s), including {legacy} legacy event(s). '
          f'Last chain digest: {m.last_hash}')
    return m


def operation(m: Machine, kind: str, h=None):
    if kind == 'state':
        return {'status': 'RUNS', 'state': m.snapshot(), 'relative_field': m.relative_snapshot(), 'ledger_head': m.last_hash}
    if kind not in ('step', 'reverse', 'transport'):
        return {'error': 'Only state, step, reverse, transport are allowed'}
    before = m.snapshot()
    trial = m.copy()
    try:
        meta = apply_mutation(trial, kind, h)
    except ValueError as e:
        return {'error': str(e), 'status': 'OPEN'}
    after = trial.snapshot()
    # Independent compiler return; never ask the model to certify it.
    check = trial.copy()
    if kind == 'step':
        check.backward()
    elif kind == 'reverse':
        check.forward()
    else:
        check.transport(h)
    assert check.snapshot() == before, f'{kind}: way home failed'
    payload = {'version': 3, 'operation': kind, 'before': before,
               'after': after, 'route': meta}
    digest = chain_hash(m.last_hash, payload)
    record = {**payload, 'prev_hash': m.last_hash, 'event_hash': digest}
    # Write before committing the in-memory state, so a disk failure cannot create
    # an unrecorded accepted transition. Existing ledger lines are never rewritten.
    with LEDGER.open('a', encoding='utf-8') as f:
        f.write(canonical(record) + '\n')
        f.flush()
        os.fsync(f.fileno())
    trial.last_hash = digest
    m.adopt(trial)
    return {'status': 'RUNS', **record, 'return_verified': True,
            'relative_field': m.relative_snapshot(),
            'native_field_key_rewrites': 0 if kind == 'transport' else 1}


def commute(m: Machine, h: int):
    """Run two commuting paths on copies; leave the live state and ledger unchanged."""
    if type(h) is not int or not 0 <= h < (1 << m.n):
        return {'status': 'OPEN', 'error': 'invalid n-bit XOR offset'}
    starting = m.snapshot()
    left = m.copy(); left.transport(h); left.forward()
    right = m.copy(); right.forward(); right.transport(h)
    assert left.snapshot() == right.snapshot(), 'commuting square failed'
    lback = left.copy(); lback.backward(); lback.transport(h)
    rback = right.copy(); rback.transport(h); rback.backward()
    assert lback.snapshot() == starting == rback.snapshot(), 'commuting return failed'
    return {'status': 'RUNS', 'identity': 'F(H_h(S)) == H_h(F(S))',
            'h': h, 'starting': starting, 'left': left.snapshot(),
            'right': right.snapshot(), 'both_returns_verified': True,
            'live_machine_unchanged': m.snapshot() == starting,
            'ledger_head': m.last_hash, 'ledger_head_unchanged': True}




def _target_machine(value, n):
    """Turn a COMPLETE external readout into a machine; never invent missing fields.

    Returns (Machine, None) or (None, structured diagnostic).
    """
    if not isinstance(value, dict):
        return None, {'status': 'OPEN', 'binding': 'UNLINKED',
                      'missing_link': 'complete target state (n,p,w,field)'}
    required = {'n', 'p', 'w', 'field'}
    missing = sorted(required - set(value))
    if missing:
        return None, {'status': 'OPEN', 'binding': 'UNLINKED',
                      'missing_fields': missing,
                      'missing_link': 'full target readout; no omitted fields are set to zero'}
    if any(type(value[key]) is not int for key in ('n','p','w')) or value['n'] != n:
        return None, {'status':'OPEN', 'binding':'UNBOUND',
                      'missing_link':'target n,p,w must be integers; width must match the active machine'}
    if not isinstance(value['field'], dict):
        return None, {'status':'OPEN', 'binding':'UNLINKED',
                      'missing_link':'field must be an explicit object, including {} if empty'}
    f = {}
    mask = (1 << n) - 1
    for key, val in value['field'].items():
        if type(val) is not int or type(key) not in (str,int):
            return None, {'status':'OPEN','binding':'UNBOUND',
                          'missing_link':'field entries must have integral address/value'}
        if isinstance(key, str) and (not key.isdecimal() or str(int(key)) != key):
            return None, {'status':'OPEN','binding':'UNBOUND',
                          'missing_link': 'field addresses must be canonical nonnegative decimal integers'}
        k = int(key)
        if not (0 <= k <= mask and 0 <= val <= mask):
            return None, {'status':'OPEN','binding':'UNBOUND',
                          'missing_link':'field address/value exceeds n-bit domain'}
        if val: f[k] = val
    try:
        return Machine(n=n, p=value['p'], w=value['w'], field=f), None
    except ValueError as exc:
        return None, {'status':'OPEN','binding':'UNBOUND','missing_link':str(exc)}


def bind(m: Machine, *, target_where=None, predecessor_of_current=False,
         predecessor_of=None, target_state=None):
    """Derive a route from the supplied HOLE rather than search ring contents.

    Contract alternatives (exactly one):
      target_where: relocate current WHAT to an absolute p using H_(p XOR target).
      predecessor_of_current=True: invert F on the current complete destination.
      predecessor_of: invert F on a supplied COMPLETE destination state.
      target_state: one-step destination; resolve ownership among H, F, F^-1.

    Distinct operators that land at the same target are reported AMBIGUOUS;
    absent or partial target fields are UNLINKED, never defaulted to zero.
    Every BOUND route includes its inverse way home. Read-only.
    """
    initial = m.snapshot(); head = m.last_hash
    goals = sum((target_where is not None, predecessor_of_current is True,
                 predecessor_of is not None, target_state is not None))
    if goals != 1 or type(predecessor_of_current) is not bool:
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':'supply exactly one: target_where, predecessor_of_current=true, predecessor_of, target_state'}
    proof = {'live_machine_unchanged': True, 'ledger_head_unchanged': True,
             'ledger_head':head}
    if target_where is not None:
        if type(target_where) is not int or not 0 <= target_where < 1 << m.n:
            return {'status':'OPEN','binding':'UNBOUND',
                    'missing_link':f'WHERE must be an n-bit integer 0..{(1<<m.n)-1}', **proof}
        h = m.p ^ target_where             # direct address: no candidate enumeration
        dest = m.copy(); dest.transport(h)
        home = dest.copy(); home.transport(h)
        assert home.snapshot() == initial and dest.w == m.w
        assert dest.relative_snapshot() == m.relative_snapshot()
        result = {'status':'RUNS','binding':'BOUND','relation':'same WHAT at required WHERE',
                  'source':initial,'destination':dest.snapshot(),
                  'route':{'operation':'transport','h':h,'cost':'O(1) native field operations'},
                  'way_home':{'operation':'transport','h':h},
                  'return_verified': True, **proof}
    elif predecessor_of_current or predecessor_of is not None:
        if predecessor_of_current:
            destination = m.copy()
        else:
            destination, error = _target_machine(predecessor_of, m.n)
            if error:
                return {**error, **proof}
        source = destination.copy(); inverse = source.backward()
        check = source.copy(); check.forward()
        assert check.snapshot() == destination.snapshot()
        result = {'status':'RUNS','binding':'BOUND','relation':'unique F-predecessor of complete destination',
                  'source':source.snapshot(),'destination':destination.snapshot(),
                  'route':{'operation':'reverse','x':inverse['x'],
                           'readout':'x=P_n(w_destination)', 'cost':'O(1) native field operations'},
                  'way_home':{'operation':'step'},'return_verified':True,**proof}
    else:
        destination, error = _target_machine(target_state, m.n)
        if error:
            return {**error,**proof}
        wanted = destination.snapshot()
        providers = []
        # Exactly three declared providers; no generated candidate words or paths.
        # H: the constraint is equality of relative computation (w,E).
        if m.w == destination.w and m.relative_snapshot() == destination.relative_snapshot():
            h = m.p ^ destination.p
            providers.append({'operation':'transport','h':h,
                              'way_home':{'operation':'transport','h':h}})
        # F: derive the required predecessor FROM the destination and compare.
        preceding = destination.copy(); preceding.backward()
        if preceding.snapshot() == initial:
            providers.append({'operation':'step','way_home':{'operation':'reverse'}})
        # F^-1: derive the required successor FROM the destination and compare.
        following = destination.copy(); following.forward()
        if following.snapshot() == initial:
            providers.append({'operation':'reverse','way_home':{'operation':'step'}})
        if len(providers) == 1:
            p = providers[0]
            result = {'status':'RUNS','binding':'BOUND',
                      'relation':'one-step provider selected by exact endpoint constraints',
                      'source':initial,'destination':wanted,'route':p,
                      'way_home':p['way_home'],'return_verified':True,**proof}
        elif len(providers) > 1:
            result = {'status':'OPEN','binding':'AMBIGUOUS',
                      'source':initial,'destination':wanted,'providers':providers,
                      'missing_link':'operator identity; an independent readout must select the provider',
                      **proof}
        else:
            result = {'status':'OPEN','binding':'UNBOUND',
                      'source':initial,'destination':wanted,'providers':[],
                      'missing_link':'no one-step H, F, or F^-1 provider owns this endpoint; a composition or another lawful operator is not yet supplied',
                      **proof}
    assert m.snapshot() == initial and m.last_hash == head, 'BIND mutated live state'
    return result


def verify_bind():
    """Exact finite tests; all are independent of LM Studio and existing user ledger."""
    checks = 0
    rng = random.Random(20261102)
    for n in range(2,17):
        mask = (1<<n)-1
        for _ in range(180):
            D={}
            for __ in range(1+rng.randrange(8)):
                k=rng.randrange(mask+1); v=rng.randrange(mask+1)
                if v: D[k]=v
            m=Machine(n, rng.randrange(mask+1), rng.randrange(mask+1), D)
            initial=m.snapshot(); h=rng.randrange(mask+1)
            a=bind(m,target_where=m.p ^ h)
            assert a['binding']=='BOUND' and a['destination']['p']==(m.p^h)
            assert a['route']['h']==h and a['return_verified']
            dest=m.copy(); dest.forward()
            b=bind(m, target_state=dest.snapshot())
            # Endpoint may have two valid one-step providers on degenerate states.
            assert any(p['operation']=='step' for p in
                       ([b['route']] if b['binding']=='BOUND' else b.get('providers',[])))
            c=bind(m,predecessor_of=dest.snapshot())
            assert c['binding']=='BOUND' and c['source']==initial
            d=bind(m,predecessor_of_current=True)
            expected=m.copy(); expected.backward()
            assert d['source']==expected.snapshot()
            assert m.snapshot()==initial
            checks+=4
    missing=bind(Machine(),target_state={'n':4,'p':1,'w':8})
    assert missing['binding']=='UNLINKED' and missing['missing_fields']==['field']
    # With u at both XOR-opposite cells, F and H_u have the same endpoint;
    # the backwards provider also agrees. A genuine multi-owner HOLE.
    probe=Machine(n=4,p=0,w=0,field={0:15,15:15})
    endpoint=probe.copy(); endpoint.forward()
    result=bind(probe,target_state=endpoint.snapshot())
    assert result['binding']=='AMBIGUOUS',result
    assert bind(Machine(),target_where=100)['binding']=='UNBOUND'
    checks+=3
    print(f'RUNS: {checks:,} BIND checks (widths 2..16, populated fields, exact inverse, ambiguous/unlinked cases).')


def request_json(method, endpoint, payload=None, timeout=180):
    headers = {'Content-Type': 'application/json'}
    if TOKEN:
        headers['Authorization'] = f'Bearer {TOKEN}'
    data = None if payload is None else json.dumps(payload).encode('utf-8')
    req = Request(BASE + endpoint, headers=headers, data=data, method=method)
    with urlopen(req, timeout=timeout) as res:
        return json.loads(res.read().decode('utf-8'))


def find_model():
    data = request_json('GET', '/api/v1/models')
    loaded = [(model['display_name'], inst['id'])
              for model in data.get('models', []) if model.get('type') == 'llm'
              for inst in model.get('loaded_instances', [])]
    if not loaded:
        print('No loaded LLM. Load Gemma 4 E4B in LM Studio, then rerun.')
        return None
    for i, (label, ident) in enumerate(loaded, 1):
        print(f'  {i}. {label} [{ident}]')
    print('Using first loaded model:', loaded[0][0])
    return loaded[0][1]


# v5: independent coupled pair compiler with exact inverse and witnessed relative returns.
class PairMachine:
    """Two readers on one field, in the same n-bit domain.

    Internal E[r]=V[r XOR s] is relative to reader A.  Both readers
    read their OLD field locations before both writes commit, as in
    NEXUS_Pair_Self_Move.c; coincident writes XOR together.
    """
    def __init__(self, n=4, pA=0, wA=2, pB=0, wB=2, field=None):
        mask = (1 << n) - 1
        if not 2 <= n <= 16:
            raise ValueError('supported prototype width is 2..16')
        if any(type(v) is not int or not 0 <= v <= mask
               for v in (pA,wA,pB,wB)):
            raise ValueError('readers and carries must fit n bits')
        D = {int(k):int(v) for k,v in (field or {}).items() if int(v)}
        if any(not (0 <= k <= mask and 0 <= v <= mask) for k,v in D.items()):
            raise ValueError('field keys and values must fit n bits')
        self.n, self.pA, self.wA, self.pB, self.wB = n,pA,wA,pB,wB
        self.V = {k ^ pA: v for k,v in D.items()}
        self.s = 0
        self.last_hash = GENESIS

    def copy(self):
        other = PairMachine(self.n, self.pA, self.wA, self.pB, self.wB)
        other.V,other.s = self.V.copy(),self.s
        other.last_hash = self.last_hash
        return other

    @staticmethod
    def _xor_write(D, key, value):
        updated = D.get(key, 0) ^ value
        if updated:
            D[key] = updated
        else:
            D.pop(key, None)

    def forward(self):
        old_s, old_A, old_B = self.s, self.pA, self.pB
        delta = old_A ^ old_B
        key_A,key_B = old_s, old_s ^ delta
        xA = self.V.get(key_A,0) ^ self.wA
        xB = self.V.get(key_B,0) ^ self.wB
        yA,yB = partner(xA,self.n), partner(xB,self.n)
        self._xor_write(self.V,key_A,yA)
        self._xor_write(self.V,key_B,yB)
        self.s, self.pA,self.pB,self.wA,self.wB = (old_s ^ xA,old_A ^ xA,old_B ^ xB,yA,yB)
        return {'xA':xA,'xB':xB,'written_A':old_A,'written_B':old_B}

    def backward(self):
        xA,xB = partner(self.wA,self.n),partner(self.wB,self.n)
        old_s,old_A,old_B = self.s ^ xA,self.pA ^ xA,self.pB ^ xB
        key_A,key_B = old_s,old_s ^ old_A ^ old_B
        self._xor_write(self.V,key_A,self.wA)
        self._xor_write(self.V,key_B,self.wB)
        old_wA,old_wB = self.V.get(key_A,0) ^ xA, self.V.get(key_B,0) ^ xB
        self.s,self.pA,self.pB,self.wA,self.wB = old_s,old_A,old_B,old_wA,old_wB
        return {'xA':xA,'xB':xB,'restored_A':old_A,'restored_B':old_B}

    def transport(self,h):
        if type(h) is not int or not 0 <= h < (1<<self.n):
            raise ValueError('invalid XOR placement offset')
        self.pA ^= h
        self.pB ^= h
        return {'h':h,'internal_field_unchanged':True}

    def swap(self):
        delta = self.pA ^ self.pB
        self.pA,self.pB = self.pB,self.pA
        self.wA,self.wB = self.wB,self.wA
        self.s ^= delta
        return {'swapped':True,'field_unchanged':True}

    def relative_field(self):
        rel={k ^ self.s:v for k,v in self.V.items()}
        return {str(k):rel[k] for k in sorted(rel)}

    def snapshot(self):
        absolute={self.pA ^ self.s ^ k:v for k,v in self.V.items()}
        return {
            'n':self.n,'pA':self.pA,'wA':self.wA,
            'pB':self.pB,'wB':self.wB,
            'field':{str(k):absolute[k] for k in sorted(absolute)}
        }

    def internal(self):
        return {'delta':self.pA ^ self.pB,'wA':self.wA,'wB':self.wB,'relative_field':self.relative_field()}


def pair_return_certificate(pair, ticks=None, *, expected_h=None, max_steps=20000):
    """Check a SUPPLIED depth; never find t by searching an orbit.

    States BOUND/AMBIGUOUS/UNBOUND/UNLINKED; providers are label-preserved
    and reader-exchanged placements. Does not modify `pair`.
    """
    if ticks is None:
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':'return depth t is not supplied; derive it independently from the pair compiler'}
    if type(ticks) is not int or ticks <= 0 or ticks > max_steps:
        return {'status':'OPEN','binding':'UNBOUND','missing_link':f'ticks must be 1..{max_steps} for this bounded verification tool; no search performed'}
    mask=(1<<pair.n)-1
    if expected_h is not None and (type(expected_h) is not int or not 0 <= expected_h <= mask):
        return {'status':'OPEN','binding':'UNBOUND','missing_link':'expected_h must be a valid n-bit word'}
    initial=pair.snapshot()
    current=pair.copy()
    for _ in range(ticks):
        current.forward()
    end=current.snapshot()
    candidates=[]
    # Providers are constructed from the required end-reader position, not
    # by enumerating address offsets or scanning intermediate readouts.
    for label_mode in ('same_labels','swapped_labels'):
        provider=pair.copy()
        if label_mode=='swapped_labels':
            provider.swap()
        h=provider.pA ^ current.pA
        if expected_h is not None and h!=expected_h:
            continue
        provider.transport(h)
        if provider.snapshot()==end and h!=0:
            candidates.append({'reader_labels':label_mode,'h':h,
                               'operation': 'transport' if label_mode=='same_labels' else 'swap_then_transport',
                               'way_home':{'operation':'inverse_pair_steps','ticks':ticks}})
    # Verify the return path by inversely replaying the SUPPLIED number of steps.
    reverse=current.copy()
    for _ in range(ticks):
        reverse.backward()
    inverse_ok = reverse.snapshot()==initial
    assert inverse_ok, 'pair reverse failed on a supplied route'
    common={'ticks':ticks,'source':initial,'endpoint':end,'return_verified':inverse_ok,
            'live_machine_unchanged':pair.snapshot()==initial,
            'field_vacuum_at_endpoint':not end['field'],
            'route_policy':'supplied-depth verification; no candidate-depth search'}
    if len(candidates)==1:
        return {'status':'RUNS','binding':'BOUND','route':candidates[0],**common}
    if len(candidates)>1:
        return {'status':'OPEN','binding':'AMBIGUOUS','providers':candidates,
                'missing_link':'reader-label ownership; an independent distinguishing readout is required',**common}
    return {'status':'OPEN','binding':'UNBOUND','providers':[],
            'missing_link':'this supplied depth does not give a nonzero same- or swapped-label transport of the source',**common}


def verify_pair():
    rng=random.Random(20261102)
    checks=0
    for n in range(2,17):
        mask=(1<<n)-1
        for _ in range(240):
            D={}
            for __ in range(rng.randrange(10)):
                key=rng.randrange(mask+1); val=rng.randrange(mask+1)
                if val: D[key]=val
            P=PairMachine(n,rng.randrange(mask+1),rng.randrange(mask+1),
                            rng.randrange(mask+1),rng.randrange(mask+1),D)
            original=P.snapshot()
            # exact inverse with filled fields, including coincident readers
            a=P.copy(); a.forward(); a.backward()
            assert a.snapshot()==original
            b=P.copy(); b.backward(); b.forward()
            assert b.snapshot()==original
            h=rng.randrange(mask+1)
            a=P.copy(); a.transport(h); a.forward()
            b=P.copy(); b.forward(); b.transport(h)
            assert a.snapshot()==b.snapshot()
            a=P.copy(); a.swap(); a.forward()
            b=P.copy(); b.forward(); b.swap()
            assert a.snapshot()==b.snapshot()
            a=P.copy(); a.swap(); a.swap()
            assert a.snapshot()==original
            a=P.copy(); a.transport(h);a.transport(h)
            assert a.snapshot()==original
            # Verify invariant native frame pA^s under steps.
            a=P.copy(); frame=a.pA ^ a.s; a.forward()
            assert (a.pA ^ a.s)==frame
            checks+=7
    # Verify tiny and moderate source-C carrier witnesses, without searching.
    n=4;r=n//2;eR=1<<(r-1);eL=1<<r;u=(1<<n)-1
    def from_k(coord):return ((eR if coord&1 else 0)^(eL if coord&2 else 0)^(u if coord&4 else 0))
    assert [partner(from_k(i),n) for i in range(8)]==[from_k((i ^ 4) if (i&1)==((i>>1)&1) else (i ^ 3)) for i in range(8)]
    a=PairMachine(n,0,eR,0,eR)
    assert pair_return_certificate(a,2)['binding']=='AMBIGUOUS'
    b=PairMachine(n,0,eR,from_k(4),from_k(6))
    result=pair_return_certificate(b,806)
    assert result['binding'] in ('BOUND','AMBIGUOUS') and result['field_vacuum_at_endpoint'],result
    assert pair_return_certificate(b)['binding']=='UNLINKED'
    checks+=4
    print(f'RUNS: {checks:,} pair compiler checks across widths 2..16; two independent source-C supplied-depth certificates.')

PAIR_LEDGER = ROOT / 'NEXUS_local_AI_pair_events.jsonl'  # new; original single-reader ledger untouched
PAIR_GENESIS = '0' * 64


def seed_pair():
    # Exact 806-step witness from NEXUS_Pair_Self_Move.c:
    # A=(0,eR), B=(u,u+eL) in the eight-word carrier at n=4.
    n=4
    u=(1<<n)-1
    eR=1<<(n//2-1)
    eL=1<<(n//2)
    return PairMachine(n, 0,eR,u,u^eL)


def pair_replay(path=None):
    path=PAIR_LEDGER if path is None else Path(path)
    pair=seed_pair()
    pair.last_hash=PAIR_GENESIS
    if not path.exists():
        return pair
    count=0
    with path.open('r',encoding='utf-8') as f:
        for count,line in enumerate(f,1):
            event=json.loads(line)
            if event.get('before')!=pair.snapshot():
                raise RuntimeError(f'Pair ledger discontinuity at line {count}')
            if event.get('prev_hash')!=pair.last_hash:
                raise RuntimeError(f'Pair ledger parent hash mismatch at line {count}')
            payload={k:v for k,v in event.items() if k not in ('prev_hash','event_hash')}
            digest=chain_hash(pair.last_hash,payload)
            if event.get('event_hash')!=digest:
                raise RuntimeError(f'Pair ledger event hash mismatch at line {count}')
            op=event['operation']
            route = pair.forward() if op=='step' else pair.backward() if op=='reverse' else None
            if route is None or route!=event['route'] or pair.snapshot()!=event['after']:
                raise RuntimeError(f'Pair ledger re-execution mismatch at line {count}')
            pair.last_hash=digest
    print(f'RUNS: replayed {count} pair event(s); pair head {pair.last_hash}')
    return pair


def pair_operation(pair,kind,path=None):
    path=PAIR_LEDGER if path is None else Path(path)
    if kind=='state':
        return {'status':'RUNS','state':pair.snapshot(),
                'internal':pair.internal(),'ledger_head':pair.last_hash}
    if kind not in ('step','reverse'):
        return {'status':'OPEN','binding':'UNBOUND','missing_link':'pair mutation requires step or reverse'}
    before=pair.snapshot()
    trial=pair.copy()
    route=trial.forward() if kind=='step' else trial.backward()
    after=trial.snapshot()
    return_machine=trial.copy()
    if kind=='step':return_machine.backward()
    else:return_machine.forward()
    assert return_machine.snapshot()==before
    payload={'version':5,'operation':kind,'before':before,'after':after,'route':route}
    digest=chain_hash(pair.last_hash,payload)
    record={**payload,'prev_hash':pair.last_hash,'event_hash':digest}
    with path.open('a',encoding='utf-8') as f:
        f.write(canonical(record)+'\n')
        f.flush();os.fsync(f.fileno())
    pair.pA,pair.wA,pair.pB,pair.wB,pair.s=(trial.pA,trial.wA,trial.pB,trial.wB,trial.s)
    pair.V=trial.V
    pair.last_hash=digest
    return {'status':'RUNS',**record,'return_verified':True}


def pair_bind_return(pair,ticks=None,expected_h=None):
    before=pair.snapshot();head=pair.last_hash
    result=pair_return_certificate(pair,ticks,expected_h=expected_h)
    assert pair.snapshot()==before and pair.last_hash==head
    return {**result,'ledger_head':head,'ledger_head_unchanged':True}


PAIR_TOOLS=[
    {'type':'function','function':{'name':'nexus_pair_state',
        'description':'READ the independent two-reader coupled field and their relative coordinate; no mutation.',
        'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_pair_step',
        'description':'Commit exactly one simultaneous coupled-reader step; records a separate SHA-256 pair ledger event.',
        'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_pair_reverse',
        'description':'Commit one exact inverse step of the independent coupled-reader state.',
        'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_pair_bind_return',
        'description':'READ-ONLY verify a GIVEN two-reader return depth; determine same-label or swapped-label self-transport and return path. '
                      'DO NOT guess, invent, or scan for a period. Supply ticks=806 for the documented n4 A=(0,eR), B=(u,u+eL) seed.',
        'parameters':{'type':'object','properties':{
            'ticks':{'type':'integer','minimum':1,'maximum':20000,
                     'description':'Required return depth established independently; do not invent.'},
            'expected_h':{'type':'integer','minimum':0,'maximum':15,
                          'description':'Optional independently specified expected XOR displacement.'}},
            'required':['ticks'],'additionalProperties':False}}},
]


def pair_model_dispatch(name,args):
    if not isinstance(args,dict):
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'expected object arguments'}
    if 'pair_machine' not in globals() or pair_machine is None:
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'pair engine not initialized'}
    contract={
        'nexus_pair_state':(set(),set()),
        'nexus_pair_step':(set(),set()),
        'nexus_pair_reverse':(set(),set()),
        'nexus_pair_bind_return':({'ticks','expected_h'},{'ticks'})
    }
    if name not in contract:
        return {'status':'OPEN','binding':'UNBOUND','missing_link':'not a pair tool'}
    allowed,required=contract[name]
    supplied=set(args)
    if not required.issubset(supplied) or not supplied.issubset(allowed):
        return {'status':'OPEN','binding':'UNLINKED','missing_fields':sorted(required-supplied),
                'unexpected_fields':sorted(supplied-allowed)}
    if name=='nexus_pair_bind_return':
        return pair_bind_return(pair_machine,ticks=args['ticks'],expected_h=args.get('expected_h'))
    return pair_operation(pair_machine,{
        'nexus_pair_state':'state','nexus_pair_step':'step','nexus_pair_reverse':'reverse'}[name])

# v4.1: explicit contracts, one task per tool, so an empty call is invalid.
# The internal bind() resolver is unchanged: no mathematical rule was edited.
STATE_SCHEMA = {
    'type': 'object',
    'properties': {
        'n': {'type': 'integer'},
        'p': {'type': 'integer'},
        'w': {'type': 'integer'},
        'field': {'type': 'object', 'additionalProperties': {'type': 'integer'}},
    },
    'required': ['n','p','w','field'],
    'additionalProperties': False,
}

TOOLS = [
    {'type':'function','function':{'name':'nexus_state',
       'description':'READ current exact state and ledger digest; no mutation.',
       'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_step',
       'description':'Execute one reversible forward reader-field step F.',
       'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_reverse',
       'description':'Execute exact inverse F^-1 of the current state.',
       'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_transport',
       'description':'Execute XOR relocation H_h on the LIVE machine; changes its position, writes a ledger event.',
       'parameters':{'type':'object','properties':{'h':{
           'type':'integer','minimum':0,'maximum':(1<<WIDTH)-1}},
           'required':['h'],'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_commute',
       'description':'Read-only verify F H_h = H_h F for the current state, without committing a change.',
       'parameters':{'type':'object','properties':{'h':{
           'type':'integer','minimum':0,'maximum':(1<<WIDTH)-1}},
           'required':['h'],'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_bind',
       'description':'READ-ONLY BIND: preserve the current WHAT and relocate its reader to a REQUIRED target WHERE. '
                     'Call with exactly {"target_where":8} if target WHERE is 8. '
                     'Derives h = current_p XOR target_where and the exact inverse route. '
                     'Do not send empty arguments or other goal fields.',
       'parameters':{'type':'object','properties':{
           'target_where': {'type':'integer','minimum':0,'maximum':(1<<WIDTH)-1,
                            'description':'REQUIRED absolute destination reader position, not XOR offset.'}},
           'required':['target_where'],'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_bind_predecessor',
       'description':'READ-ONLY BIND: derive the unique F-predecessor of the CURRENT complete state.',
       'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_bind_predecessor_of',
       'description':'READ-ONLY BIND: derive the unique F-predecessor of a supplied COMPLETE destination state. '
                     'All four fields n,p,w,field are REQUIRED; never assume empty omitted field.',
       'parameters':{'type':'object','properties':{'target_state':STATE_SCHEMA},
                     'required':['target_state'],'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_bind_endpoint',
       'description':'READ-ONLY BIND: identify the one-step provider (transport, F, or F^-1) '
                     'for a supplied COMPLETE target state. May return BOUND, AMBIGUOUS, or UNBOUND.',
       'parameters':{'type':'object','properties':{'target_state':STATE_SCHEMA},
                     'required':['target_state'],'additionalProperties':False}}},
]

TOOLS.extend(PAIR_TOOLS)

SYSTEM = (
    'You interface with an independently executing NEXUS compiler. '
    'Use actual tools; never invent results. '
    'When the user requests preserving current WHAT at destination WHERE, '
    'call nexus_bind with required argument target_where equal to that absolute address. '
    'For current predecessor use nexus_bind_predecessor. '
    'For a supplied complete endpoint use nexus_bind_endpoint. '
    'Do not pass mutually exclusive goals together. '
    'Do not invent missing readout fields or choose an operator if AMBIGUOUS. '
    'BIND is read-only. Explain tool result and its way home, without claiming the live state changed. '
    'For the bound PAIR, use nexus_pair_bind_return with an independently supplied ticks; '
    'the compiler verifies but does not discover the period. Never invent tick counts.'
)

def dispatch_model_tool(machine, name, args):
    """Exact gateway. Invalid args are OPEN and never mutate the machine."""
    if not isinstance(args, dict):
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'tool arguments must be a JSON object'}
    if name.startswith('nexus_pair_'):
        return pair_model_dispatch(name,args)
    contracts = {
        'nexus_state': (set(),set()),
        'nexus_step': (set(),set()),
        'nexus_reverse': (set(),set()),
        'nexus_transport': ({'h'},{'h'}),
        'nexus_commute': ({'h'},{'h'}),
        'nexus_bind': ({'target_where'},{'target_where'}),
        'nexus_bind_predecessor': (set(),set()),
        'nexus_bind_predecessor_of': ({'target_state'},{'target_state'}),
        'nexus_bind_endpoint': ({'target_state'},{'target_state'}),
    }
    if name not in contracts:
        return {'status':'OPEN','error':'unknown compiler tool '+str(name)}
    allowed, required = contracts[name]
    supplied = set(args)
    if not required.issubset(supplied) or not supplied.issubset(allowed):
        return {
            'status':'OPEN','binding':'UNLINKED',
            'missing_fields': sorted(required-supplied),
            'unexpected_fields': sorted(supplied-allowed),
            'missing_link':'invalid tool arguments for '+name+'; required fields '+str(sorted(required)),
            'received_args': args,
        }
    if name == 'nexus_bind':
        return bind(machine, target_where=args['target_where'])
    if name == 'nexus_bind_predecessor':
        return bind(machine, predecessor_of_current=True)
    if name == 'nexus_bind_predecessor_of':
        return bind(machine, predecessor_of=args['target_state'])
    if name == 'nexus_bind_endpoint':
        return bind(machine, target_state=args['target_state'])
    if name == 'nexus_commute':
        return commute(machine, args['h'])
    return operation(machine, {'nexus_state':'state','nexus_step':'step',
                              'nexus_reverse':'reverse','nexus_transport':'transport'}[name],
                     args.get('h'))


def ask_model(model_id, machine, user_text, history=None):
    # Retains the earlier calling convention. Print actual tool arguments to disambiguate failures.
    messages = history if history is not None else [{'role':'system','content':SYSTEM}]
    messages.append({'role':'user','content':user_text})
    for _ in range(16):
        response = request_json('POST', '/v1/chat/completions', {
            'model': model_id, 'messages': messages, 'tools': TOOLS,
            'tool_choice':'auto', 'temperature':0.1, 'max_tokens':1024,
            'stream':False,
        })
        msg = response['choices'][0]['message']
        calls = msg.get('tool_calls') or []
        if not calls:
            content = msg.get('content') or '(no text returned)'
            messages.append({'role':'assistant','content':content})
            print('\nMODEL:',content)
            return content
        messages.append({'role':'assistant','content':msg.get('content'), 'tool_calls':calls})
        for call in calls:
            fn = call.get('function',{}).get('name','')
            raw_args = call.get('function',{}).get('arguments') or '{}'
            print(f'  CALL {fn} raw arguments: {raw_args}')
            try:
                args = json.loads(raw_args)
                result = dispatch_model_tool(machine,fn,args)
            except (ValueError,TypeError,json.JSONDecodeError) as err:
                result = {'status':'OPEN','binding':'UNLINKED','missing_link':'invalid tool JSON: '+str(err)}
            print(f'  TOOL {fn}:',canonical(result))
            messages.append({'role':'tool','tool_call_id':call['id'],'content':canonical(result)})
    print('Tool-round budget reached; exact machine state:',canonical(machine.snapshot()))

def main():
    global pair_machine
    verify()
    verify_bind()
    verify_pair()
    pair_machine=pair_replay()
    print('BIND prediction SHA-256:', BIND_PREDICTION_SHA256)
    machine = replay()
    print('Current state:', canonical(machine.snapshot()))
    print('Pair seed/current:', canonical(pair_machine.snapshot()))
    print('LM Studio:', BASE)
    try:
        model_id = find_model()
    except (URLError, HTTPError, TimeoutError, ValueError) as err:
        print('LM Studio not reachable:', err)
        model_id = None
    chat = [{'role':'system','content':SYSTEM}]
    print('Commands: :state, :step, :reverse, :transport <h>, :commute <h>, :bind <WHERE>, :back, :pair-state, :pair-step, :pair-reverse, :pair-return <ticks>, :clear, :quit')
    while True:
        try:
            line = input('nexus> ').strip()
        except (EOFError, KeyboardInterrupt):
            break
        if not line: continue
        if line == ':quit': break
        if line in (':pair-state',':pair-step',':pair-reverse'):
            print(json.dumps(pair_operation(pair_machine, line[len(':pair-'):]), indent=2))
            continue
        if line.startswith(':pair-return'):
            parts=line.split()
            try:
                ticks=int(parts[1]) if len(parts)==2 else None
                print(json.dumps(pair_bind_return(pair_machine,ticks=ticks),indent=2))
            except ValueError as err:
                print('Invalid ticks:',err)
            continue
        if line == ':clear':
            chat[:] = [{'role':'system','content':SYSTEM}]
            print('Conversation context reset; deterministic machine unchanged.')
        elif line in (':state',':step',':reverse'):
            print(json.dumps(operation(machine,line[1:]),indent=2))
        elif line == ':back':
            print(json.dumps(bind(machine,predecessor_of_current=True),indent=2))
        elif line.startswith(':bind '):
            try:
                print(json.dumps(bind(machine,target_where=int(line.split()[1],0)),indent=2))
            except (ValueError,IndexError) as err:
                print('Invalid WHERE:',err)
        elif line.startswith(':transport') or line.startswith(':commute'):
            parts = line.split()
            if len(parts) != 2:
                print('Usage: :transport 3 or :commute 3  (also accepts 0x3)')
                continue
            try:
                h = int(parts[1],0)
                print(json.dumps(commute(machine,h) if parts[0] == ':commute'
                                 else operation(machine,'transport',h),indent=2))
            except ValueError as err:
                print('Invalid transport:', err)
        elif not model_id:
            print('Model unavailable; compiler commands still work.')
        else:
            try:
                ask_model(model_id,machine,line,chat)
            except (URLError,HTTPError,TimeoutError,KeyError,ValueError) as err:
                print('LM Studio API error:',err)



# Jupyter startup: no input() loop. Both engines keep separate ledgers.
verify()
verify_bind()
verify_pair()
machine = replay()
pair_machine = pair_replay()
print("\nNEXUS PAIR v5 READY — two isolated compiler states")
print("Single reader:", canonical(machine.snapshot()))
print("Paired readers:", canonical(pair_machine.snapshot()))
print("NEXUS pair ledger:", str(PAIR_LEDGER))
print("LM Studio:", BASE)
print("To verify the supplied 806-step pair return, use: pair_bind_return(pair_machine,806)")


RUNS: 30,844 deterministic conditions; P, arbitrary-field roundtrip, transport involution/group law, F H = H F, inverse commutation, and 1000-step return at n=4,8,16.
RUNS: 10,803 BIND checks (widths 2..16, populated fields, exact inverse, ambiguous/unlinked cases).
RUNS: 25,204 pair compiler checks across widths 2..16; two independent source-C supplied-depth certificates.
Replayed 19 event(s), including 4 legacy event(s). Last chain digest: d3c28760a59dc2aa6f4de437de31d217e0e9e976af823b3c183a047cb1513077

NEXUS PAIR v5 READY — two isolated compiler states
Single reader: {"field":{"4":1,"5":8},"n":4,"p":12,"w":1}
Paired readers: {"field":{},"n":4,"pA":0,"pB":15,"wA":2,"wB":11}
NEXUS pair ledger: D:\@User Data\Downloads\NEXUS_local_AI_pair_events.jsonl
LM Studio: http://localhost:1235
To verify the supplied 806-step pair return, use: pair_bind_return(pair_machine,806)


## Check the pair, then ask Gemma

Run the commands below in a **new cell**. The return certificate does not modify either machine.

```python
print(pair_operation(pair_machine,"state"))
print(pair_bind_return(pair_machine,806))
model_id = find_model()
if model_id:
    ask_model(model_id, machine,
      "Read the bound pair. Verify the independently supplied 806-step return, identify the displacement and its way home; do not modify either state.")
```

For a committed paired forward/reverse run: `pair_operation(pair_machine,"step")` then `pair_operation(pair_machine,"reverse")`. These append events to the **separate** paired ledger.

BIND statuses: BOUND (one provider), AMBIGUOUS (both label assignments), UNBOUND (no matching translated provider at supplied depth), UNLINKED (depth not supplied). A supplied depth is **verified**, not derived.


In [2]:
print(pair_operation(pair_machine,"state"))
print(pair_bind_return(pair_machine,806))
model_id = find_model()
if model_id:
    ask_model(model_id, machine,
      "Read the bound pair. Verify the independently supplied 806-step return, identify the displacement and its way home; do not modify either state.")

{'status': 'RUNS', 'state': {'n': 4, 'pA': 0, 'wA': 2, 'pB': 15, 'wB': 11, 'field': {}}, 'internal': {'delta': 15, 'wA': 2, 'wB': 11, 'relative_field': {}}, 'ledger_head': '0000000000000000000000000000000000000000000000000000000000000000'}
{'status': 'RUNS', 'binding': 'BOUND', 'route': {'reader_labels': 'same_labels', 'h': 9, 'operation': 'transport', 'way_home': {'operation': 'inverse_pair_steps', 'ticks': 806}}, 'ticks': 806, 'source': {'n': 4, 'pA': 0, 'wA': 2, 'pB': 15, 'wB': 11, 'field': {}}, 'endpoint': {'n': 4, 'pA': 9, 'wA': 2, 'pB': 6, 'wB': 11, 'field': {}}, 'return_verified': True, 'live_machine_unchanged': True, 'field_vacuum_at_endpoint': True, 'route_policy': 'supplied-depth verification; no candidate-depth search', 'ledger_head': '0000000000000000000000000000000000000000000000000000000000000000', 'ledger_head_unchanged': True}
  1. Gemma 4 E4B [google/gemma-4-e4b]
Using first loaded model: Gemma 4 E4B
  CALL nexus_pair_state raw arguments: {}
  TOOL nexus_pair_state: {"

In [3]:

model_id = find_model()

if model_id:
    ask_model(
        model_id,
        machine,
        "Call nexus_pair_bind_return with ticks=806. "
        "Use the compiler result to report the translation, "
        "reader ownership, endpoint, and inverse. "
        "Do not modify the live state."
    )


  1. Gemma 4 E4B [google/gemma-4-e4b]
Using first loaded model: Gemma 4 E4B
  CALL nexus_pair_bind_return raw arguments: {"ticks":806}
  TOOL nexus_pair_bind_return: {"binding":"BOUND","endpoint":{"field":{},"n":4,"pA":9,"pB":6,"wA":2,"wB":11},"field_vacuum_at_endpoint":true,"ledger_head":"0000000000000000000000000000000000000000000000000000000000000000","ledger_head_unchanged":true,"live_machine_unchanged":true,"return_verified":true,"route":{"h":9,"operation":"transport","reader_labels":"same_labels","way_home":{"operation":"inverse_pair_steps","ticks":806}},"route_policy":"supplied-depth verification; no candidate-depth search","source":{"field":{},"n":4,"pA":0,"pB":15,"wA":2,"wB":11},"status":"RUNS","ticks":806}

MODEL: The compiler result indicates a successful binding (`"binding":"BOUND"`), meaning a unique path exists between the source and the endpoint for the given return depth of 806 ticks. The live state remains unchanged throughout this read-only operation.

Here is the bre